# Modul 7a: Skills nutzen — Fähigkeiten installieren

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du eine Skill-Registry mit Routing-Logik.

🎯 **Lernziel:** Skills als modulare Erweiterungen verstehen, eine Registry bauen und Intent-basiertes Routing implementieren.

---

## Skills = Modulare Fähigkeiten

Ein Skill ist eine klar definierte Fähigkeit mit:
- **Trigger:** Wann wird er aktiviert? (Keywords, Patterns)
- **Instruction:** Was soll der Agent tun?
- **Scripts:** Optional: Shell-Scripts für Ausführung

```
User-Nachricht → Intent erkennen → Skill matchen → Ausführen
```

Qin et al. (2023) systematisieren Tool Learning in 4 Phasen:
**Discovery → Selection → Usage → Creation** (Modul 7b)

In [ ]:
# Skill-Registry mit Intent-Routing

class Skill:
    """Eine einzelne Agent-Fähigkeit."""
    def __init__(self, name, description, triggers, instruction, execute_fn=None):
        self.name = name
        self.description = description
        self.triggers = triggers  # Liste von Keywords/Patterns
        self.instruction = instruction
        self.execute_fn = execute_fn or (lambda ctx: f'Skill {name} ausgeführt.')
        self.installed = False

    def install(self):
        self.installed = True
        print(f'  ✅ Skill "{self.name}" installiert.')

    def matches(self, user_input):
        """Prüft ob der User-Input zu diesem Skill passt."""
        input_lower = user_input.lower()
        return any(t.lower() in input_lower for t in self.triggers)

    def run(self, context):
        if not self.installed:
            return f'Skill "{self.name}" ist nicht installiert!'
        return self.execute_fn(context)


class SkillRegistry:
    """Verwaltet alle verfügbaren Skills."""
    def __init__(self):
        self.skills = {}

    def register(self, skill):
        self.skills[skill.name] = skill

    def install(self, name):
        if name in self.skills:
            self.skills[name].install()
        else:
            print(f'  ❌ Skill "{name}" nicht gefunden!')

    def route(self, user_input):
        """Findet den passenden Skill für eine Nachricht."""
        matches = []
        for skill in self.skills.values():
            if skill.installed and skill.matches(user_input):
                matches.append(skill)
        return matches

    def show(self):
        print('\n📋 Skill-Registry:')
        for s in self.skills.values():
            status = '✅' if s.installed else '⬜'
            print(f'  {status} {s.name}: {s.description}')
            print(f'      Triggers: {s.triggers}')


# === Skills definieren ===
print('=== Skills registrieren ===\n')
registry = SkillRegistry()

registry.register(Skill(
    name='deep-research',
    description='Tiefgehende Web-Recherche zu einem Thema',
    triggers=['recherchiere', 'deep research', 'research', 'untersuche'],
    instruction='Führe eine strukturierte Recherche durch: 1) Quellen sammeln, 2) Zusammenfassen, 3) Bewerten',
    execute_fn=lambda ctx: f'Recherche zu "{ctx}" gestartet... [3 Quellen gefunden, Zusammenfassung erstellt]'
))

registry.register(Skill(
    name='content-creator',
    description='LinkedIn/Blog Posts erstellen',
    triggers=['linkedin', 'post erstellen', 'content', 'blog'],
    instruction='Erstelle einen Post: Hook → Story → Takeaway → CTA',
    execute_fn=lambda ctx: f'Post erstellt:\n  Hook: Wusstest du...\n  Story: {ctx}\n  CTA: Was denkst du?'
))

registry.register(Skill(
    name='kosten-check',
    description='Preisvergleich und Kostenanalyse',
    triggers=['was kostet', 'preisvergleich', 'kosten', 'budget'],
    instruction='Recherchiere Preise, vergleiche 3 Optionen, gib Empfehlung',
    execute_fn=lambda ctx: f'Kostenanalyse für "{ctx}":\n  Option A: $10/mo\n  Option B: $25/mo\n  Empfehlung: Option A für den Start'
))

registry.register(Skill(
    name='security-audit',
    description='Sicherheitsüberprüfung des Systems',
    triggers=['security', 'sicherheit', 'audit', 'sicher'],
    instruction='Prüfe: Gesperrte Pfade, Policies, Prompt-Injection-Schutz',
    execute_fn=lambda ctx: 'Security Audit:\n  ✅ Pfade gesperrt\n  ⚠️ Policy zu offen\n  ❌ Kein Injection-Schutz'
))

registry.show()

# Skills installieren
print('\n=== Installation ===\n')
registry.install('deep-research')
registry.install('content-creator')
registry.install('kosten-check')
# security-audit absichtlich NICHT installiert

registry.show()

In [ ]:
# Intent-Routing in Aktion

test_messages = [
    'Recherchiere mal zum Thema Agent-Architekturen',
    'Erstelle einen LinkedIn Post über AI Agents',
    'Was kostet ein Cloud-Server?',
    'Mach einen Security Audit',       # Nicht installiert!
    'Wie wird das Wetter morgen?',     # Kein Skill dafür
    'Deep Research zu MCP Protocol',
]

print('=== Routing-Test ===\n')
for msg in test_messages:
    matches = registry.route(msg)
    print(f'💬 "{msg}"')
    if matches:
        for skill in matches:
            print(f'   → Skill: {skill.name}')
            result = skill.run(msg)
            print(f'   → Ergebnis: {result}')
    else:
        print(f'   → ❌ Kein passender Skill (Fallback an LLM)')
    print()

In [ ]:
# 🎯 ÜBUNG: Eigenen Skill erstellen und registrieren
#
# Aufgabe:
# 1. Erstelle einen 'wetter'-Skill mit passenden Triggers
# 2. Registriere und installiere ihn
# 3. Teste dass 'Wie wird das Wetter morgen?' jetzt geroutet wird

# TODO: Skill definieren
# my_skill = Skill(
#     name='wetter',
#     description='...',
#     triggers=[...],
#     instruction='...',
#     execute_fn=lambda ctx: '...'
# )

# TODO: Registrieren und installieren
# registry.register(my_skill)
# registry.install('wetter')

# TODO: Testen
# matches = registry.route('Wie wird das Wetter morgen?')
# print(matches)

In [ ]:
# ✅ LÖSUNG

wetter_skill = Skill(
    name='wetter',
    description='Wettervorhersage abfragen',
    triggers=['wetter', 'temperatur', 'regnet', 'sonnig'],
    instruction='Wetter abfragen und kurz zusammenfassen',
    execute_fn=lambda ctx: f'Wetter-Abfrage: Berlin 18°C, sonnig mit Wolken. (Simulation)'
)

registry.register(wetter_skill)
registry.install('wetter')

print('\n=== Test: Wetter-Routing ===')
for msg in ['Wie wird das Wetter morgen?', 'Regnet es heute?', 'Recherchiere zum Wetter']:
    matches = registry.route(msg)
    print(f'\n💬 "{msg}"')
    for skill in matches:
        print(f'   → {skill.name}: {skill.run(msg)}')

## Skill-Routing in der Praxis

| Absicht | Skill | Trigger |
|---------|-------|---------|
| "Recherchiere X" | deep-research | recherchiere, research |
| "LinkedIn Post" | content-creator | linkedin, post, content |
| "Was kostet X?" | kosten-check | kosten, preisvergleich |
| "Security Check" | security-audit | security, audit |

**Precedence:** Wenn mehrere Skills matchen, gewinnt der spezifischere (mehr Trigger-Matches).

---

### ✅ Self-Check
- [ ] Ich kann erklären was ein Skill ist (Trigger + Instruction + Script)
- [ ] Ich verstehe Intent-basiertes Routing
- [ ] Ich kann einen eigenen Skill registrieren und testen

**→ Weiter zu [Modul 7b: Skills bauen](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul7b_skills_bauen.ipynb)**